In [ ]:
# Detectable rate from the CORSIKA showers:  the analytic spectrum, each shower's own detectable area
import nbformat
import pandas as pd
exec(nbformat.read("full_analytic.ipynb", as_version=4).cells[0].source)

SAMPLE = "finecore"
#SAMPLE = "thinh"          # lighter thinning (THINH), ~10 showers per step
showers = pd.read_feather(f"../showers/tables/{SAMPLE}.feather")
showers["E_eV"] = showers.E_GeV * 1e9
print(f"{SAMPLE}:  {len(showers)} showers")

In [ ]:
# The energy steps:  every neighbouring pair stands in the same ratio (to rounding)
energies = np.sort(showers.E_eV.unique())
ratios = energies[1:] / energies[:-1]
ratio = ratios.mean()
assert ratios.max() / ratios.min() - 1 < 1e-5, "energy steps not in a constant ratio"
DLNE = np.log(ratio)        # bin width in ln(E/E0), for any E0
print(f"{len(energies)} steps, {energies[0]:.3g} to {energies[-1]:.3g} eV, ratio {ratio:.6f}")

In [ ]:
# Rate at Phi_D = 50:  sum over energy steps of (flux in the step) x (mean detectable area of its showers)
area = showers.a_det50_m2              # detectable area of each shower, m^2

rate = 0.0
for E in energies:
    flux = E * dFdE(E) * DLNE * YEAR        # flux on the ground in this step, per m^2 per year
    mean_area = area[showers.E_eV == E].mean()
    rate += flux * mean_area

print(f"Phi_D 50:  {rate:.2f} per year   (analytic {R(50) * YEAR:.2f})")

In [ ]:
# Rate at Phi_D = 20:  sum over energy steps of (flux in the step) x (mean detectable area of its showers)
area = showers.a_det20_m2              # detectable area of each shower, m^2

rate = 0.0
for E in energies:
    flux = E * dFdE(E) * DLNE * YEAR        # flux on the ground in this step, per m^2 per year
    mean_area = area[showers.E_eV == E].mean()
    rate += flux * mean_area

print(f"Phi_D 20:  {rate:.2f} per year   (analytic {R(20) * YEAR:.2f})")

In [ ]:
# Contributing energies at Phi_D = 50:  each energy step's share of the rate, against the analytic model
area = showers.a_det50_m2              # detectable area of each shower, m^2

contribution = []
for E in energies:
    flux = E * dFdE(E) * DLNE * YEAR        # flux on the ground in this step, per m^2 per year
    mean_area = area[showers.E_eV == E].mean()
    contribution.append(flux * mean_area)   # this step's contribution to the rate, per year

E = np.logspace(np.log10(E_min(50)), 20, 400)
analytic = dRdlnE(E, 50) * DLNE * YEAR      # the analytic rate in a step of the same width

p1 = plt.plot(energies, contribution, "o", label=f"CORSIKA {SAMPLE}")
p2 = plt.plot(E, analytic, label="analytic")
i = np.argmax(contribution)
plt.annotate(f"R = {sum(contribution):.1f} / yr", xy=(energies[i], contribution[i]), xytext=(50, 25),
             textcoords="offset points", color=p1[0].get_color(),
             arrowprops=dict(arrowstyle="->", color=p1[0].get_color()))
j = np.argmax(analytic)
plt.annotate(f"R = {R(50) * YEAR:.1f} / yr", xy=(E[j], analytic[j]), xytext=(20, 35),
             textcoords="offset points", color=p2[0].get_color(),
             arrowprops=dict(arrowstyle="->", color=p2[0].get_color()))
plt.ylim(0, 1.3 * plt.ylim()[1])
plt.xscale("log")
plt.xlabel("E (eV)")
plt.ylabel("rate per energy step (per year)")
plt.title("Contributing energies, $\\Phi_D$ = 50")
plt.legend()
plt.savefig("pdf/corsika_contributing_energies.pdf")
plt.show()